# Models

A key component of `synthpop` is the concept of a **model**. Models describe the physical properties of a galaxy population and define the assumptions used to generate synthetic galaxies.

For stellar populations, a model typically specifies:

- The stellar mass function
- The star formation history
- The chemical enrichment (metallicity) history
- Dust attenuation

Both `synthpop` and `synthesizer` provide the machinery required to construct and work with these models. Together, they enable users to generate physically motivated galaxy populations and predict their observable properties.

`synthpop` also includes a variety of pre-made models.

In [1]:
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from synthesizer.parametric import SFH, Stars, ZDist
from synthpop.distribution_functions import Schechter
from synthpop.models import Model
from unyt import yr, Myr, Msun, Gyr, unyt_quantity, Mpc, dimensionless


## Galaxy Stellar Mass Function

In [2]:

# Define the galaxy stellar mass function (GSMF) using a Schechter function
x_star = 10**10.745 * Msun     
phi_star = 10**(-2.437) * Mpc**-3  
alpha = -1.465
gsmf = Schechter(x_star=x_star, alpha=alpha, phi_star=phi_star)

## Star formation and metal enrichment history

In [3]:

# Define a delta function for metallicity
metal_dist_function = ZDist.DeltaConstant

metal_dist_parameters = {
    "log10metallicity": -2.5
}


age_of_universe = unyt_quantity(cosmo.age(0).value, str(cosmo.age(0).unit))

sfh_function = SFH.LogNormal

def peak_age_function(mass, age_of_Universe=1.37E10 * yr):

    value = (mass/(1E9*Msun))**2.5 * 1E9 + np.random.normal(0, 1e9)

    return np.min((value, age_of_Universe.to('yr').value)) * yr

def tau_function(mass):
    return np.clip(np.random.normal(0.6, 0.1) * dimensionless, 0.1, 1.0)

sfh_parameters = {
    "tau": 0.6 * dimensionless,
    "peak_age": 1e10 * yr,
    "max_age": age_of_universe
}

# sfh_parameters = {
#     "tau": tau_function,
#     "peak_age": peak_age_function,
#     "max_age": age_of_universe
# }


sfh_function = SFH.Constant

sfh_parameters = {
    "max_age": 10000 * Myr
}


## Dust attenuation

## Create model